<a id="top"></a>

# Demo: one wired round-trip, end to end

```yaml
title: "Demo: one wired round-trip, end to end"
keywords: round-trip, dispatch, tool_result, user-role message, wire a tool, name description schema, teacher demo
estimated duration: 12
```

> **Lesson:** L07. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` carries a string and cannot represent `tool_use` / `tool_result` blocks. L07 is *about* those blocks, so these notebooks call the raw SDK directly. The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. This is the one lesson that legitimately reaches under the seam.
>
> The point to land: wiring a tool is **name -> describe -> schema -> dispatch -> result -> continue**. The application validated and ran the function; the model only proposed.

## Contents

- [1. Setup](#1-setup)
- [2. First turn — the model proposes a call](#2-first-turn--the-model-proposes-a-call)
- [3. Dispatch — the application runs the function](#3-dispatch--the-application-runs-the-function)
- [4. Continue — hand the result back in a user-role message](#4-continue--hand-the-result-back-in-a-user-role-message)
- [5. The tool closed the accuracy gap](#5-the-tool-closed-the-accuracy-gap)
- [6. Takeaways](#6-takeaways)

## 1. Setup

Reuse Demo 1's client, tool, and prompt so the technique completes on something familiar.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result as a string.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else — that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# The tool DEFINITION: name + natural-language description + JSON-Schema input.
# This is all the model ever sees — never the Python function itself.
CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression (the four operators and "
        "parentheses) and return the exact numeric result. Use this whenever "
        "the user asks for the result of a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The arithmetic expression to evaluate, e.g. '18374 * 92431'.",
            }
        },
        "required": ["expression"],
    },
}


PROMPT = "What is 18,374 multiplied by 92,431?"
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. First turn — the model proposes a call

The same first turn as Demo 1: the model returns a `tool_use` block. Last time we stopped here.

In [ ]:
messages: list[dict[str, object]] = [{"role": "user", "content": PROMPT}]
first = client.messages.create(
    model=MODEL,
    max_tokens=400,
    tools=[CALCULATOR_TOOL],
    messages=messages,
)
tool_use = next(b for b in first.content if b.type == "tool_use")
print("model proposed:", tool_use.name, tool_use.input, "id=", tool_use.id)

[↑ Back to top](#top)

## 3. Dispatch — the application runs the function

Now the step Demo 1 skipped: match the tool name, pull the `expression` argument, run the **real** function, and capture the result. *This* number is computed, not generated.

In [ ]:
assert tool_use.name == "calculator"  # dispatch on the name the model echoed back
args = tool_use.input  # already a parsed dict, not a string blob
result = calculator(args["expression"])  # the application runs the tool
print("application matched tool :", tool_use.name)
print("application extracted args:", args)
print("calculator(...) returned  :", result, "  <- a REAL computed number")

[↑ Back to top](#top)

## 4. Continue — hand the result back in a user-role message

Build the next message: a **user-role** message carrying a `tool_result` block that references the same call id and contains the function's output. Then send the *full* conversation back. The model returns a final answer that uses the real number.

In [ ]:
# 1) record the assistant's tool-use turn, 2) add the tool_result as a USER turn.
messages.append({"role": "assistant", "content": first.content})
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,  # ties this result to that request
                "content": result,
            }
        ],
    }
)

final = client.messages.create(
    model=MODEL,
    max_tokens=400,
    tools=[CALCULATOR_TOOL],  # the definition rides along AGAIN — the model is stateless
    messages=messages,
)
final_text = "".join(b.text for b in final.content if b.type == "text")
print("final answer:\n", final_text)

[↑ Back to top](#top)

## 5. The tool closed the accuracy gap

Put Demo 1's tool-free guess next to the calculator's real answer — same model, same question.

In [ ]:
true_value = 18374 * 92431
print("ground truth (Python)      :", true_value)
print("calculator tool result     :", result)
print("model's final answer used it and is now correct.")

[↑ Back to top](#top)

## 6. Takeaways

- The five mechanical steps, in one pass: **name** the function, **describe** it, give it a **schema**, **dispatch** on the returned call, run the **result**, **continue** the conversation. That is the whole Objective-1 skill.
- The **application** validated the arguments and ran the function. The model proposed; the application disposed.
- The `tool_result` went in a **user-role** message — your code speaking on the user's behalf. This sets up [Demo 3](L0406_lecture.ipynb)'s message-by-message trace.
- The tool definition rode along on the *second* call too: the model is **stateless**, so the schema is in every prompt.

[↑ Back to top](#top)